In [3]:
import os
import yaml
import h5py
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

# 1. Setup Paths
repo_root = os.path.abspath(os.path.join(os.getcwd(), ".."))
clean_meta_path = os.path.join(repo_root, "data", "clean", "metadata_clean.parquet")
raw_wave_path = os.path.join(repo_root, "data", "raw", "waveforms.hdf5")
ai_ready_dir = os.path.join(repo_root, "data", "ai_ready")
os.makedirs(ai_ready_dir, exist_ok=True)

df = pd.read_parquet(clean_meta_path)
print(f"Loaded {len(df)} clean metadata records.")

# --- DATA SAMPLE SPECIFICATION ---
# Input: 3-component ground motion waveform trace array of shape (3, 1500)
# Target: Earth science target (source_magnitude float32)

print("\n--- B1: DIVERSITY (INDEPENDENT UNITS) ---")
unique_stations = df["station_code"].nunique() if "station_code" in df.columns else df["station_network_code"].nunique()
unique_events = df["event_id"].nunique() if "event_id" in df.columns else len(df)
unique_networks = df["station_network_code"].nunique() if "station_network_code" in df.columns else 1

print(f"Independent Stations: {unique_stations}")
print(f"Independent Events: {unique_events}")
print(f"Independent Networks: {unique_networks}")

print("\n--- B2: DISTRIBUTION SUMMARY ---")
if "preferred_source_magnitude" in df.columns:
    mag_col = "preferred_source_magnitude"
elif "source_magnitude" in df.columns:
    mag_col = "source_magnitude"
else:
    mag_col = df.select_dtypes(include=[np.number]).columns[0]

mag_stats = df[mag_col].describe()
print(f"Target Variable ({mag_col}) Stats:")
print(mag_stats)

print("\n--- B3: NOISE ASSESSMENT (SNR ESTIMATION) ---")
if "trace_snr_db" in df.columns:
    # Convert string SNR values to numeric floats, coercing bad values to NaN
    snr_vals = pd.to_numeric(df["trace_snr_db"], errors="coerce").dropna()
    if len(snr_vals) > 0:
        print(f"Trace SNR (dB) - Mean: {snr_vals.mean():.2f} dB, Median: {snr_vals.median():.2f} dB")
    else:
        print("SNR column contained no valid numeric entries; estimating from trace samples...")
else:
    print("Estimating SNR from waveform traces...")
    sample_snrs = []
    with h5py.File(raw_wave_path, "r") as hf:
        data_grp = hf["data"]
        for _, row in df.head(100).iterrows():
            b_val, t_idx = str(row["bucket"]).strip(), int(row["tindex"])
            b_key = b_val if b_val.startswith("bucket") else f"bucket{b_val}"
            if b_key in data_grp and t_idx < data_grp[b_key].shape[0]:
                raw_tr = data_grp[b_key][t_idx]
                tr = np.array(raw_tr, dtype=np.float32)
                signal_power = np.mean(tr**2)
                noise_window = tr[:, :100] if tr.ndim == 2 and tr.shape[1] >= 100 else tr[:100]
                noise_power = np.mean(noise_window**2) + 1e-8
                snr_db = 10 * np.log10((signal_power + 1e-8) / noise_power)
                sample_snrs.append(snr_db)
                
    if len(sample_snrs) > 0:
        print(f"Sample Estimated SNR (dB) - Mean: {np.mean(sample_snrs):.2f} dB")

print("\n--- B4: AUGMENTATION DEFINITIONS ---")
print("Physically Valid Augmentation: Random time-shift along time axis (preserves wave arrivals & phase alignment).")
print("Physics-Breaking Augmentation: Shuffling time samples across channels independently (destroys wave polarization & phase velocity).")

print("\n--- B5: LABEL PROVENANCE ---")
print("Labels produced by PNSN analyst picks and catalog magnitude calculations.")

print("\n--- B6: DIMENSIONALITY ---")
N = len(df)
D = 3 * 1500  # 3 channels x 1500 samples
approx_bytes = N * D * 4  # float32 = 4 bytes
print(f"Dataset Dimensions -> N (samples): {N}, D (features per trace): {D}")
print(f"Approximate Raw Tensor Size: {approx_bytes / (1024**2):.2f} MB")

print("\n--- B7: INDEPENDENT PARTITIONING (STATION HOLDOUT) ---")

# Determine available station identifier column
if "station_code" in df.columns:
    stn_col = "station_code"
elif "station_network_code" in df.columns:
    stn_col = "station_network_code"
else:
    stn_col = None

if stn_col and df[stn_col].nunique() > 1:
    # Convert unique values strictly to a python list of strings
    stations = df[stn_col].dropna().astype(str).unique().tolist()
    
    train_stns, test_stns = train_test_split(stations, test_size=0.20, random_state=42)
    val_stns, test_stns = train_test_split(test_stns, test_size=0.50, random_state=42)

    train_set, val_set = set(train_stns), set(val_stns)

    def assign_split(stn):
        stn_str = str(stn)
        if stn_str in train_set:
            return "train"
        elif stn_str in val_set:
            return "val"
        else:
            return "test"
    df["partition"] = df[stn_col].apply(assign_split)
else:
    indices = df.index.tolist()
    train_idx, test_idx = train_test_split(indices, test_size=0.20, random_state=42)
    val_idx, test_idx = train_test_split(test_idx, test_size=0.50, random_state=42)
    
    df["partition"] = "train"
    df.loc[val_idx, "partition"] = "val"
    df.loc[test_idx, "partition"] = "test"

print("Partition distribution:")
print(df["partition"].value_counts())

# Save Partition Index
partition_file = os.path.join(ai_ready_dir, "partitions.parquet")
df[["trace_name", "partition"]].to_parquet(partition_file, index=False)
print(f"Saved partitions to: {partition_file}")

# Update description.yml dynamically with final counts
desc_path = os.path.join(ai_ready_dir, "description.yml")
if os.path.exists(desc_path):
    with open(desc_path, "r") as f:
        desc_data = yaml.safe_load(f) or {}
    
    desc_data["number_of_independent_units"] = {
        "stations": int(unique_stations),
        "events": int(unique_events),
        "networks": int(unique_networks)
    }
    
    with open(desc_path, "w") as f:
        yaml.dump(desc_data, f, default_flow_style=False)
    print(f"Updated {desc_path} with accurate unit counts.")

Loaded 18640 clean metadata records.

--- B1: DIVERSITY (INDEPENDENT UNITS) ---
Independent Stations: 281
Independent Events: 8055
Independent Networks: 6

--- B2: DISTRIBUTION SUMMARY ---
Target Variable (preferred_source_magnitude) Stats:
count    18640.000000
mean         1.322097
std          0.727969
min          0.000000
25%          0.800000
50%          1.300000
75%          1.800000
max          4.800000
Name: preferred_source_magnitude, dtype: float64

--- B3: NOISE ASSESSMENT (SNR ESTIMATION) ---
SNR column contained no valid numeric entries; estimating from trace samples...

--- B4: AUGMENTATION DEFINITIONS ---
Physically Valid Augmentation: Random time-shift along time axis (preserves wave arrivals & phase alignment).
Physics-Breaking Augmentation: Shuffling time samples across channels independently (destroys wave polarization & phase velocity).

--- B5: LABEL PROVENANCE ---
Labels produced by PNSN analyst picks and catalog magnitude calculations.

--- B6: DIMENSIONALITY 